# EchoFactory NB-04: Evaluasi & Export ONNX
1. Load model per mesin
2. Hitung anomaly score (cosine distance dari centroid normal)
3. Evaluasi AUC dan pAUC
4. Export ke ONNX untuk FastAPI EchoFactory
5. Simpan inference_config.json


In [ ]:
import os
import json
import math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import roc_auc_score, roc_curve
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
MACHINE_TYPES = ['fan', 'pump', 'slider', 'valve']
FEAT_DIR = '/kaggle/working/features'
EMBED_DIM = 256
print(f'Device: {device}')


In [ ]:
class ConvBNPReLU(nn.Module):
    def __init__(self, ic, oc, k=3, s=1, p=1, g=1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(ic, oc, k, s, p, groups=g, bias=False),
            nn.BatchNorm2d(oc),
            nn.PReLU(oc)
        )
    def forward(self, x): return self.net(x)

class DepthwiseSep(nn.Module):
    def __init__(self, ic, oc, s=1):
        super().__init__()
        self.net = nn.Sequential(
            ConvBNPReLU(ic, ic, s=s, g=ic),
            ConvBNPReLU(ic, oc, k=1, p=0)
        )
    def forward(self, x): return self.net(x)

class MobileFaceNet(nn.Module):
    def __init__(self, ed=256):
        super().__init__()
        self.enc = nn.Sequential(
            ConvBNPReLU(1, 32, s=2), DepthwiseSep(32, 64),
            DepthwiseSep(64, 128, s=2), DepthwiseSep(128, 128),
            DepthwiseSep(128, 256, s=2), DepthwiseSep(256, 256),
            DepthwiseSep(256, 512, s=2), nn.AdaptiveAvgPool2d(1)
        )
        self.head = nn.Sequential(nn.Flatten(), nn.Linear(512, ed), nn.BatchNorm1d(ed))
    def forward(self, x): return self.head(self.enc(x))

class STgramMFN(nn.Module):
    def __init__(self, nc, ed=256):
        super().__init__()
        self.mel = MobileFaceNet(ed)
        self.tgram = MobileFaceNet(ed)
        self.fuse = nn.Sequential(nn.Linear(ed*2, ed), nn.BatchNorm1d(ed), nn.PReLU(ed))
        self._placeholder = nn.Parameter(torch.zeros(nc, ed))
    def forward(self, mel, tg):
        feat = self.fuse(torch.cat([self.mel(mel), self.tgram(tg)], dim=1))
        return F.normalize(feat, dim=1)

print('Model classes defined OK')


In [ ]:
@torch.no_grad()
def get_embeddings(model, feat_dir, machine, condition, bs=128):
    data = torch.load(os.path.join(feat_dir, f'{machine}_{condition}.pt'))
    mel_feats = data['features'][:, 0:1]
    tg_feats = data['features'][:, 1:2]
    dl = DataLoader(TensorDataset(mel_feats, tg_feats), batch_size=bs, shuffle=False)
    model.eval()
    embs = [model(mb.to(device), tb.to(device)).cpu() for mb, tb in dl]
    return torch.cat(embs)

def anomaly_score(normal_embs, test_embs):
    centroid = F.normalize(normal_embs.mean(dim=0, keepdim=True), dim=1)
    sims = (F.normalize(test_embs, dim=1) @ centroid.T).squeeze(1)
    return (1.0 - sims).numpy()

def compute_pauc(y_true, scores, max_fpr=0.1):
    fpr, tpr, _ = roc_curve(y_true, scores)
    mask = fpr <= max_fpr
    return float(np.trapz(tpr[mask], fpr[mask]) / max_fpr)

print('Helper functions OK')


In [ ]:
results = {}
thresholds = {}

for machine in MACHINE_TYPES:
    model_path = f'/kaggle/working/stgram_mfn_{machine}.pt'
    if not os.path.exists(model_path):
        print(f'Skip {machine}: model tidak ditemukan')
        continue

    ck = torch.load(model_path, map_location=device)
    model = STgramMFN(ck['n_classes'], ck['embed_dim']).to(device)
    model.load_state_dict(ck['model_state'])
    model.eval()

    n_embs = get_embeddings(model, FEAT_DIR, machine, 'normal')
    a_embs = get_embeddings(model, FEAT_DIR, machine, 'abnormal')

    n_scores = anomaly_score(n_embs, n_embs)
    a_scores = anomaly_score(n_embs, a_embs)

    all_scores = np.concatenate([n_scores, a_scores])
    all_labels = np.concatenate([np.zeros(len(n_scores)), np.ones(len(a_scores))])

    auc_val = roc_auc_score(all_labels, all_scores)
    pauc_val = compute_pauc(all_labels, all_scores)

    fpr, tpr, thr = roc_curve(all_labels, all_scores)
    best_thr = float(thr[np.argmax(tpr - fpr)])

    results[machine] = {'auc': float(auc_val), 'pauc': float(pauc_val)}
    thresholds[machine] = best_thr

    print(f'{machine:8s}: AUC={auc_val:.4f} | pAUC={pauc_val:.4f} | threshold={best_thr:.4f}')

with open('/kaggle/working/thresholds.json', 'w') as f:
    json.dump(thresholds, f, indent=2)
with open('/kaggle/working/auc_scores.json', 'w') as f:
    json.dump(results, f, indent=2)
print('Thresholds & AUC scores saved.')


In [ ]:
n_m = len(results)
fig, axes = plt.subplots(1, n_m, figsize=(5 * n_m, 5))
if n_m == 1:
    axes = [axes]
fig.suptitle('ROC Curve - STgram-MFN', fontsize=13, fontweight='bold')

for ax, (machine, res) in zip(axes, results.items()):
    ck = torch.load(f'/kaggle/working/stgram_mfn_{machine}.pt', map_location=device)
    m = STgramMFN(ck['n_classes'], ck['embed_dim']).to(device)
    m.load_state_dict(ck['model_state'])
    ne = get_embeddings(m, FEAT_DIR, machine, 'normal')
    ae = get_embeddings(m, FEAT_DIR, machine, 'abnormal')
    ns = anomaly_score(ne, ne)
    as_ = anomaly_score(ne, ae)
    scores = np.concatenate([ns, as_])
    labels = np.concatenate([np.zeros(len(ns)), np.ones(len(as_))])
    fpr, tpr, _ = roc_curve(labels, scores)
    ax.plot(fpr, tpr, 'b-', lw=2, label=f'AUC={res["auc"]:.3f}')
    ax.plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.5)
    mask = fpr <= 0.1
    ax.fill_between(fpr[mask], tpr[mask], alpha=0.2, color='orange', label=f'pAUC={res["pauc"]:.3f}')
    ax.set_title(machine.upper(), fontweight='bold')
    ax.set_xlabel('FPR')
    ax.set_ylabel('TPR')
    ax.legend()
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('/kaggle/working/roc_curves.png', dpi=150, bbox_inches='tight')
plt.show()

aucs = [r['auc'] for r in results.values()]
print(f'Rata-rata AUC: {np.mean(aucs):.4f}')


In [ ]:
print('Export model ke ONNX...')
for machine in results:
    ck = torch.load(f'/kaggle/working/stgram_mfn_{machine}.pt', map_location='cpu')
    m = STgramMFN(ck['n_classes'], ck['embed_dim'])
    m.load_state_dict(ck['model_state'])
    m.eval()

    dm = torch.randn(1, 1, 128, 128)
    dt = torch.randn(1, 1, 128, 128)
    onnx_path = f'/kaggle/working/stgram_mfn_{machine}.onnx'

    torch.onnx.export(
        m, (dm, dt), onnx_path,
        input_names=['mel', 'tgram'],
        output_names=['embedding'],
        dynamic_axes={
            'mel': {0: 'batch'},
            'tgram': {0: 'batch'},
            'embedding': {0: 'batch'}
        },
        opset_version=13,
        do_constant_folding=True,
        verbose=False
    )
    mb = os.path.getsize(onnx_path) / (1024 ** 2)
    print(f'  {machine}: {onnx_path} ({mb:.1f} MB)')

print('Export ONNX selesai!')


In [ ]:
cfg = {
    'model_type': 'STgram-MFN',
    'embed_dim': EMBED_DIM,
    'img_size': 128,
    'sample_rate': 16000,
    'mel_params': {'n_mels': 128, 'n_fft': 1024, 'hop_length': 512},
    'tgram_params': {'n_mels': 128, 'n_fft': 512, 'hop_length': 512},
    'thresholds': thresholds,
    'auc_scores': results,
    'models': {m: f'stgram_mfn_{m}.onnx' for m in results},
    'kaggle_dataset': 'https://www.kaggle.com/datasets/bisheshgiri/mimii-dataset'
}
with open('/kaggle/working/inference_config.json', 'w') as f:
    json.dump(cfg, f, indent=2)

print('NB-04 SELESAI!')
print('\nSemua output di /kaggle/working/:')
for fn in sorted(os.listdir('/kaggle/working/')):
    try:
        size = f"{os.path.getsize('/kaggle/working/' + fn) / (1024**2):.1f} MB"
    except Exception:
        size = ''
    print(f'  {fn:<50s} {size}')
print('\nModel siap untuk FastAPI EchoFactory!')
